# Install Dependencies


*   `transformers`: It allows you to download the model and tokenizer from the Hugging Face Hub. It also includes the Trainer API, which automates the complex math of the training loop (calculating loss, backpropagation, and updating weights).

*   `datasets`: It handles your Modified_SQL_Dataset.csv. Unlike standard Python lists or Pandas, it uses "memory mapping," which means it doesn't load the entire dataset into your RAM at once, preventing Colab from crashing.

* `accelearte`:In Colab, it handles the device placement. It ensures the model is correctly moved to the Tesla T4 GPU and manages "Mixed Precision" training (using 16-bit instead of 32-bit for math), which speeds up training and saves memory.

* `peft`: A full Llama model has billions of parameters. Fine-tuning all of them would require 10x more memory than Colab provides. peft implements LoRA (Low-Rank Adaptation), which freezes the main model and only trains a tiny "adapter" layer (less than 1% of the total parameters). This makes fine-tuning possible on consumer hardware.

* `bitsandbytes`: It shrinks the model's weight files. By default, models load in 16-bit or 32-bit. bitsandbytes compresses them into 4-bit (QLoRA). Without this, even a small Llama-3-8B model would be too large to even load into the Colab GPU memory, let alone train it.



In [1]:
!pip install -q -U transformers datasets accelerate peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 17.1 MB/s eta 0:00:00


# Load and Prepare the CSV

In [2]:
import pandas as pd
from datasets import Dataset

# Load your local CSV
df = pd.read_csv("Modified_SQL_Dataset.csv")

# Ensure columns are named 'text' and 'labels' for the trainer
df = df.rename(columns={'Query': 'text', 'Label': 'labels'})

# Convert to Hugging Face Dataset format
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

# Load Llama with 4-bit Quantization
We use `BitsAndBytesConfig` to compress the model so it fits on the T4 GPU.

In [6]:
from huggingface_hub import login
login()

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "meta-llama/Llama-3.1-8B" # Or "meta-llama/Meta-Llama-3-8B" (needs more memory)

# 1. Quantization Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# 2. Load Tokenizer & Model
tokenizer = AutoTokenizer.from_pretrained(model_id)

tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=2, # 0: Non-malicious, 1: Malicious
    quantization_config=bnb_config,
    device_map="auto",
    token=True
)
model.config.pad_token_id = tokenizer.pad_token_id

# 3. LoRA Config (The "Tuning" part)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"], # Target specific layers
    lora_dropout=0.05,
    task_type="SEQ_CLS"
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Llama-3.1-8B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# Fine-Tuning
Using the Trainer API makes the process stable on Colab.

In [12]:
from transformers import TrainingArguments, Trainer

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

training_args = TrainingArguments(
    output_dir="./llama-sql-detector",
    per_device_train_batch_size=4, # Keep small for Colab
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    report_to= "none", # To stop the Wandb prompt
    optim="paged_adamw_32bit", # Memory efficient optimizer
    save_strategy="epoch",
    fp16=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
)

trainer.train()

Map:   0%|          | 0/24735 [00:00<?, ? examples/s]

Map:   0%|          | 0/6184 [00:00<?, ? examples/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,5.993800
20,0.527800
30,0.026400
40,0.006800
50,0.085800
60,0.000200
70,0.000000
80,0.314500
90,0.093800
100,0.273700


KeyboardInterrupt: 